In [1]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


In [2]:
# Adding Display functionality of Databricks 
exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

Databricks-style helpers ready: display(), dbutils.fs/widgets/notebook, %run_notebook


In [3]:
from pyspark.sql.functions import *

# Average Review Rating

## Difficulty
Easy

## Topics
- PySpark
- SQL
- Aggregation
- Group By
- Average
- Sorting

## Problem Statement

You are given a PySpark DataFrame named `reviews` containing product reviews.

### Dataset: `reviews`

| Column | Data Type | Description |
|---|---|---|
| `review_id` | Integer | Unique identifier for each review |
| `product_id` | Integer | Identifier of the product being reviewed |
| `user_id` | Integer | Identifier of the user who wrote the review |
| `rating` | Integer | Product rating, ranging from 1 to 5 |
| `review_date` | String | Date when the review was submitted in `YYYY-MM-DD` format |

## Task

Calculate the **average rating for each product**.

The result should:

1. Group reviews by `product_id`.
2. Calculate the average of `rating` for each product.
3. Round the average rating to **2 decimal places**.
4. Return only:
   - `product_id`
   - `avg_rating`
5. Sort the result:
   - First by `avg_rating` in **descending order**.
   - Then by `product_id` in **ascending order** when two products have the same average rating.

## Expected Output

| product_id | avg_rating |
|---:|---:|
| 103 | 4.67 |
| 101 | 4.50 |
| 102 | 3.00 |

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)



In [6]:
# Sample dataset

data = [
    (1, 101, 1, 5, "2023-01-10"),
    (2, 101, 2, 4, "2023-01-11"),
    (3, 102, 3, 3, "2023-01-12"),
    (4, 102, 4, 3, "2023-01-13"),
    (5, 103, 5, 5, "2023-01-14"),
    (6, 103, 6, 5, "2023-01-15"),
    (7, 103, 7, 4, "2023-01-16")
]

schema = StructType([
    StructField("review_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("user_id", IntegerType(), False),
    StructField("rating", IntegerType(), False),
    StructField("review_date", StringType(), False)
])

reviews = spark.createDataFrame(data, schema)

reviews.show()
reviews.printSchema()

+---------+----------+-------+------+-----------+
|review_id|product_id|user_id|rating|review_date|
+---------+----------+-------+------+-----------+
|        1|       101|      1|     5| 2023-01-10|
|        2|       101|      2|     4| 2023-01-11|
|        3|       102|      3|     3| 2023-01-12|
|        4|       102|      4|     3| 2023-01-13|
|        5|       103|      5|     5| 2023-01-14|
|        6|       103|      6|     5| 2023-01-15|
|        7|       103|      7|     4| 2023-01-16|
+---------+----------+-------+------+-----------+

root
 |-- review_id: integer (nullable = false)
 |-- product_id: integer (nullable = false)
 |-- user_id: integer (nullable = false)
 |-- rating: integer (nullable = false)
 |-- review_date: string (nullable = false)



In [7]:
reviews.createOrReplaceTempView("reviews")

In [24]:
spark.sql(
         """SELECT 
                product_id, 
                ROUND(AVG(rating),2) AS average_rating 
            from reviews 
            group by product_id
            order by average_rating DESC, product_id ASC """
        )\
.show()

+----------+--------------+
|product_id|average_rating|
+----------+--------------+
|       103|          4.67|
|       101|           4.5|
|       102|           3.0|
+----------+--------------+



In [26]:
reviews\
    .groupBy("product_id")\
    .agg(
        round(avg(col("rating")),2).alias("average_rating")
    )\
    .orderBy(
        col("average_rating").desc(),col("product_id").asc()
    )\
.show()

+----------+--------------+
|product_id|average_rating|
+----------+--------------+
|       103|          4.67|
|       101|           4.5|
|       102|           3.0|
+----------+--------------+

